The relevant imports:

In [14]:
import time
from datetime import date, timedelta
import requests
import pandas as pd
import osmnx as ox

Collecting the dates:

In [ ]:
# Generating a list of relevant dates
def date_range_list(start, end):
    days = []
    cur = start
    
    while cur <= end:
        days.append(cur)
        cur += timedelta(days=1)
        
    return days

# Collecting Dates from the last 30 days
end_date = date.today()
start_date = end_date - timedelta(days=29)
all_days = date_range_list(start_date, end_date)

API Request and getting bird obseravation data for the time range:

In [ ]:
region = "DE-SH"

headers = {"X-eBirdApiToken": eBird_Key}

all_rows = []

# Collecting all the data in the relevant time range
for d in all_days:
    year = str(d.year)
    month = str(d.month).zfill(2)
    day = str(d.day).zfill(2)

    url = "https://api.ebird.org/v2/data/obs/" + region + "/historic/" + year + "/" + month + "/" + day

    r = requests.get(url, headers=headers)
    
    all_rows.extend(r.json())
    time.sleep(0.8)  
    
bird = pd.DataFrame(all_rows)

Keeping relevant columns:

In [ ]:
# Rename relevant columns
birds = bird.rename(columns={
    "comName": "species",
    "lng": "lon",
    "obsDt": "date"
})

# Keep only relevant columns
birds = birds[["species","lat","lon","date","locId","locName"]]

# §LLM Help: for Remove dublicates
locations = birds[["locId","locName","lat","lon"]].drop_duplicates().reset_index(drop=True)

Defining landuse categories for urban:

In [35]:
urban_types = ["residential","commercial","industrial","retail"]

tags = {"landuse": urban_types}   

Classification urban and rural:

In [50]:
# §LLM Help: for classification of a location as urban and rural
def classify_location(lat, lon):

    try:
        gdf = ox.features_from_point((lat, lon), tags=tags, dist=100)

        if gdf is not None and not gdf.empty:
            return "urban"
        else:
            return "rural"

    except Exception as e:

        if "No matching features" in str(e):
            return "rural"

        print("Error", lat, lon, "->", e)
        return "unknown"

In [ ]:
# Classification of each location
labels = []

for i, row in locations.iterrows():

    label = classify_location(row["lat"], row["lon"])
    labels.append(label)

    time.sleep(0.2)

locations["area_type"] = labels

Merging Data:

In [ ]:
# Merging area_type into birds dataset
birds = birds.merge(
    locations[["locId", "area_type"]],
    on="locId",
    how="left"
)

In [ ]:
Calculate Species Richness:

In [ ]:
# §LLM Help: for calculating species richness and reset index to species richness 
richness = (
    birds
    .groupby(["area_type", "locId","locName"])["species"]
    .nunique()
    .reset_index(name="species_richness")
)

In [65]:
birds.to_csv("rq_3_birds_data_classification.csv", index=False)
locations.to_csv("rq_3_locations_classified.csv", index=False)
richness.to_csv("rq_3_richness_sh_data.csv", index=False)